In [23]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn import datasets, ensemble
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.utils.fixes import parse_version
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [24]:
"""
Step 1: Load the data and Divide Sensor data from from labels
"""
df = pd.read_excel(
    "../../data/data_Hw3_new.xlsx", 
    parse_dates=["timestamp"]
)

# Captor data
X = df .iloc[:, 1:-1]
# True values
y = df .iloc[:, -1]

# Number of samples to use for training, first 4 weeks (24*7*4)
n_train = 672

In [25]:
"""
Step 2: Calculate the average day for the first month
"""
# Get all predicted data
# Predicted for the first month
y_first_month = np.array(y[:n_train])
# Shape it as a matrix
y_first_month = y_first_month.reshape(int(n_train / 24), 24)
# Get the mean day
mean_day = np.mean(y_first_month, axis = 0)

In [26]:
"""
Step 3: calculate the difference between each day and the mean day
"""
# Get the difference between every day and the mean day
rows = []
for day in y_first_month:
    rows.append(day - mean_day)
y_first_month_difference = np.array(rows).transpose()

In [28]:
"""
Step 4: Calculate the SVD from the difference days
"""

# SVD decomposition
U, S, V = np.linalg.svd(y_first_month_difference)

In [29]:
"""
Step 5: Calculate the optimal r
"""
beta = len(V)/len(U)
w_beta = 0.56*(beta**3) - 0.95*(beta**2) + 1.82*(beta) + 1.43
median_singular = np.median(S)
threshold = w_beta*median_singular
upper = 0
for s in S:
    if s > threshold:
        upper = upper + 1
#U_r = U[:, :upper]
upper

4

In [30]:
"""
Step 6: Train a model using only the first days
"""

# Data to train and testthe second model
X_train = X.iloc[:n_train]
X_test  = X.iloc[n_train:]

y_train = y.iloc[:n_train]
y_test  = y.iloc[n_train:]

# train the second model
param_grid = {'max_depth': list(range(1, 15))}
gbr = ensemble.GradientBoostingRegressor(n_estimators = 500, min_samples_split = 5, learning_rate = 0.01, random_state=0)

grid = GridSearchCV(gbr, param_grid, cv=5, scoring='r2', n_jobs=-1, return_train_score=True)
grid.fit(X_train, y_train)

print("Best params:",)
print("Best CV r2:", grid.best_score_)

best_model = grid.best_estimator_
# final evaluation on the untouched test set:
test_r2 = best_model .score(X_test, y_test)
print("Test R2:", test_r2)

Best params:
Best CV r2: 0.7126279320763205
Test R2: 0.6763300060338904


In [31]:
"""
Step 7: Denoise the data for the days left
"""

# Get the y predicted by the second model
y_predicted = best_model.predict(X)
# Get all but the first month
y_test_not_denoised = y_predicted[n_train:]
print("R2: ", r2_score(y_test, y_test_not_denoised))
# Reshape it as N*24
y_daily = np.array(y_test_not_denoised).reshape(int(len(y_test_not_denoised) / 24), 24)
# Denoise the days
Y_denoised_days = []
for day in y_daily:
    Y_denoised_days.append( mean_day + (U_r @ U_r.transpose() @ ( day - mean_day)))

# Put them back as a single vector
Y_denoised_days = np.array(Y_denoised_days)
Y_denoised_days_vec = Y_denoised_days.reshape(len(Y_denoised_days) * 24)
Y_denoised_days_vec.shape

R2:  0.6763300060338904


(2472,)

In [32]:
"""
Step 8: Calculare R² for each r
"""
for i in range (24):
    U_r = U[:, :i]
    # Get the y predicted by the second model
    y_predicted = best_model.predict(X)
    # Get all but the first month
    y_test_not_denoised = y_predicted[n_train:]
    # Reshape it as N*24
    y_daily = np.array(y_test_not_denoised).reshape(int(len(y_test_not_denoised) / 24), 24)
    # Denoise the days
    Y_denoised_days = []
    for day in y_daily:
        Y_denoised_days.append( mean_day + (U_r @ U_r.transpose() @ ( day - mean_day)))
    
    # Put them back as a single vector
    Y_denoised_days = np.array(Y_denoised_days)
    Y_denoised_days_vec = Y_denoised_days.reshape(len(Y_denoised_days) * 24)
    Y_denoised_days_vec.shape
    r2 = r2_score(y_test, Y_denoised_days_vec)
    print("iteration: ", i, "  R2: ",r2)  

iteration:  0   R2:  0.1133282668455583
iteration:  1   R2:  0.3551103859079242
iteration:  2   R2:  0.5262749586471653
iteration:  3   R2:  0.5678217175739965
iteration:  4   R2:  0.5960722585760159
iteration:  5   R2:  0.6115994220781656
iteration:  6   R2:  0.6272301211390766
iteration:  7   R2:  0.6309267568342283
iteration:  8   R2:  0.6375478495822318
iteration:  9   R2:  0.6394485938978548
iteration:  10   R2:  0.6471237698224378
iteration:  11   R2:  0.6504521882758318
iteration:  12   R2:  0.663003932393916
iteration:  13   R2:  0.6658402154402308
iteration:  14   R2:  0.6705509187579415
iteration:  15   R2:  0.671128715775682
iteration:  16   R2:  0.6733332665956495
iteration:  17   R2:  0.6735739163492063
iteration:  18   R2:  0.6749678636392904
iteration:  19   R2:  0.6749862025953817
iteration:  20   R2:  0.6750593659571797
iteration:  21   R2:  0.6752761054096984
iteration:  22   R2:  0.6765210101229382
iteration:  23   R2:  0.6752971328855306
